In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/achiver07/data-project/sc_val.jsonl
/kaggle/input/datasets/achiver07/data-project/acts_val.jsonl
/kaggle/input/datasets/achiver07/data-project/acts_train.jsonl
/kaggle/input/datasets/achiver07/data-project/tribunals_val.jsonl
/kaggle/input/datasets/achiver07/data-project/test.json
/kaggle/input/datasets/achiver07/data-project/sc_train.jsonl
/kaggle/input/datasets/achiver07/data-project/tribunals_train.jsonl
/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/sc_train.bin
/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/acts_train.bin
/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/acts_val.bin
/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/tribunals_val_f.bin
/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/tokenizer.json
/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/tribunals_train_f.bin
/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/test.json
/kaggle/input/datasets/achiver07/phase1-token

In [2]:
# ================================================================
# ARCHITECTURE 4 — HYBRID GQA DECODER TRANSFORMER
# FAST ~195M T4 VERSION
# ================================================================
#
# Decoder-only Transformer
#
# 12 layers
# d_model       = 1024
# Q heads       = 16
# KV heads      = 4
# GQA           = 16Q / 4KV
# RoPE
# SwiGLU
# Pre-LN
# Local causal attention W=256
# Full causal attention every 3rd layer
# Final LayerNorm
# Tied input/output embeddings
#
# No pretrained model
# No pretrained weights
# Pure PyTorch
#
# ------------------------------------------------
# TRAINING
# ------------------------------------------------
#
# Target:
#     16,384 tokens / optimizer step
#
# Preferred:
#     Batch 8
#     Grad accumulation 2
#
# Automatic fallback if B8 OOM:
#     Batch 4
#     Grad accumulation 4
#
# Total:
#     10,000 steps
#     163.84M tokens
#
# ------------------------------------------------
# SPEED
# ------------------------------------------------
#
# Fused QKV
# Fused SwiGLU
# Native GQA
# Fused AdamW when available
# torch.compile(reduce-overhead)
# Activation checkpointing OFF
#
# ------------------------------------------------
# IMPORTANT CUDAGRAPH FIX
# ------------------------------------------------
#
# torch.compiler.cudagraph_mark_step_begin()
# is called once at the start of each logical
# training iteration / optimizer step.
#
# This is important because each optimizer step
# contains multiple compiled forward/backward calls
# due to gradient accumulation.
#
# ================================================================


# ================================================================
# 1. IMPORTS
# ================================================================

import os
import math
import json
import time
import random
import gc

import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F


# ================================================================
# 2. CONFIG
# ================================================================

CFG = {

    # ------------------------------------------------------------
    # DATA
    # ------------------------------------------------------------

    "DATA_DIR":
        "/kaggle/input/datasets/achiver07/"
        "phase1-tokens-v40k-clean1",

    "VOCAB_SIZE": 40000,


    # ------------------------------------------------------------
    # MODEL
    # ------------------------------------------------------------

    "D_MODEL": 1024,

    "N_LAYERS": 12,

    "N_Q_HEADS": 16,

    "N_KV_HEADS": 4,

    "D_FF": 3328,

    "MAX_SEQ_LEN": 1024,

    "SEQ_LEN": 1024,

    "WINDOW": 256,

    "GLOBAL_EVERY": 3,

    "ROPE_BASE": 10000.0,

    "DROPOUT": 0.0,

    "INIT_STD": 0.02,


    # ------------------------------------------------------------
    # TRAINING
    # ------------------------------------------------------------

    # Preferred: B8 / accumulation 2.
    # Automatically falls back to B4 / accumulation 4.

    "BATCH_SIZE": 8,

    "GRAD_ACCUM": 2,

    "MAX_STEPS": 10000,

    "LR": 3e-4,

    "MIN_LR_RATIO": 0.10,

    "WARMUP_STEPS": 500,

    "WEIGHT_DECAY": 0.1,

    "GRAD_CLIP": 1.0,


    # ------------------------------------------------------------
    # PERFORMANCE
    # ------------------------------------------------------------

    "USE_FUSED_ADAMW": True,

    "USE_COMPILE": True,

    "COMPILE_MODE": "reduce-overhead",


    # ------------------------------------------------------------
    # VALIDATION
    # ------------------------------------------------------------

    "EVAL_EVERY": 500,

    "EVAL_BATCHES": 30,


    # ------------------------------------------------------------
    # CHECKPOINT
    # ------------------------------------------------------------

    "SAVE_EVERY": 500,

    "OUTPUT_DIR":
        "/kaggle/working/arch4_hybrid_gqa_195m",


    # ------------------------------------------------------------
    # RANDOM SEED
    # ------------------------------------------------------------

    "SEED": 42,
}


# ================================================================
# 3. CLEAN CUDA STATE
# ================================================================

gc.collect()

if torch.cuda.is_available():

    torch.cuda.empty_cache()

    torch.cuda.reset_peak_memory_stats()


# ================================================================
# 4. DEVICE
# ================================================================

DEVICE = torch.device(

    "cuda"
    if torch.cuda.is_available()
    else "cpu"

)


print("Device:", DEVICE)


if DEVICE.type == "cuda":

    props = torch.cuda.get_device_properties(0)

    print(
        "GPU:",
        props.name
    )

    print(
        "GPU memory:",
        f"{props.total_memory / 1024**3:.2f} GB"
    )


# ================================================================
# 5. PERFORMANCE SETTINGS
# ================================================================

torch.set_float32_matmul_precision(
    "high"
)


# ================================================================
# 6. SEED
# ================================================================

def seed_everything(seed):

    random.seed(seed)

    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():

        torch.cuda.manual_seed_all(seed)


seed_everything(
    CFG["SEED"]
)


# ================================================================
# 7. PATHS
# ================================================================

DATA_DIR = CFG["DATA_DIR"]


PATHS = {

    "acts_train":
        f"{DATA_DIR}/acts_train.bin",

    "sc_train":
        f"{DATA_DIR}/sc_train.bin",

    "tribunals_train":
        f"{DATA_DIR}/tribunals_train_f.bin",

    "acts_val":
        f"{DATA_DIR}/acts_val.bin",

    "sc_val":
        f"{DATA_DIR}/sc_val.bin",

    "tribunals_val":
        f"{DATA_DIR}/tribunals_val_f.bin",

    "tokenizer":
        f"{DATA_DIR}/tokenizer.json",

    "test":
        f"{DATA_DIR}/test.json",
}


print("\nChecking files...")


for name, path in PATHS.items():

    print(
        f"{name:20s}",
        os.path.exists(path),
        path
    )


os.makedirs(

    CFG["OUTPUT_DIR"],

    exist_ok=True

)


# ================================================================
# 8. TOKENIZER
# ================================================================

print("\nLoading tokenizer...")


with open(

    PATHS["tokenizer"],

    "r",

    encoding="utf-8"

) as f:

    tokenizer_json = json.load(f)


print(
    "Tokenizer loaded."
)


# ================================================================
# 9. BIN DATA
# ================================================================

def load_tokens(path):

    return np.memmap(

        path,

        dtype=np.uint16,

        mode="r",

    )


train_arrays = {

    "acts":
        load_tokens(
            PATHS["acts_train"]
        ),

    "sc":
        load_tokens(
            PATHS["sc_train"]
        ),

    "tribunals":
        load_tokens(
            PATHS["tribunals_train"]
        ),

}


val_arrays = {

    "acts":
        load_tokens(
            PATHS["acts_val"]
        ),

    "sc":
        load_tokens(
            PATHS["sc_val"]
        ),

    "tribunals":
        load_tokens(
            PATHS["tribunals_val"]
        ),

}


print("\nTraining data:")


for name, arr in train_arrays.items():

    print(

        f"{name:12s}: "
        f"{len(arr):,} tokens"

    )


print("\nValidation data:")


for name, arr in val_arrays.items():

    print(

        f"{name:12s}: "
        f"{len(arr):,} tokens"

    )


# ================================================================
# 10. SOURCE MIXTURE
# ================================================================

SOURCE_NAMES = [

    "acts",
    "sc",
    "tribunals",

]


SOURCE_PROBS = np.array(

    [
        0.273,
        0.469,
        0.258,
    ],

    dtype=np.float64,

)


SOURCE_PROBS /= SOURCE_PROBS.sum()


print(

    "\nSource probabilities:",

    dict(
        zip(
            SOURCE_NAMES,
            SOURCE_PROBS
        )
    )

)


# ================================================================
# 11. RANDOM WINDOW SAMPLER
# ================================================================

def sample_batch(

    arrays,
    batch_size,
    seq_len,

):

    xs = np.empty(

        (
            batch_size,
            seq_len,
        ),

        dtype=np.int64,

    )


    ys = np.empty(

        (
            batch_size,
            seq_len,
        ),

        dtype=np.int64,

    )


    source_indices = np.random.choice(

        len(SOURCE_NAMES),

        size=batch_size,

        p=SOURCE_PROBS,

    )


    for b, source_idx in enumerate(

        source_indices

    ):

        source_name = SOURCE_NAMES[

            source_idx

        ]


        data = arrays[

            source_name

        ]


        max_start = (

            len(data)

            -

            seq_len

            -

            1

        )


        start = np.random.randint(

            0,

            max_start + 1,

        )


        xs[b] = data[

            start:

            start + seq_len

        ]


        ys[b] = data[

            start + 1:

            start + seq_len + 1

        ]


    return (

        torch.from_numpy(xs),

        torch.from_numpy(ys),

    )


# ================================================================
# 12. RoPE
# ================================================================

class RotaryEmbedding(nn.Module):

    def __init__(

        self,
        dim,
        max_seq_len,
        base=10000.0,

    ):

        super().__init__()


        assert dim % 2 == 0


        inv_freq = (

            1.0
            /
            (
                base
                **
                (
                    torch.arange(

                        0,

                        dim,

                        2,

                        dtype=torch.float32,

                    )

                    /

                    dim

                )

            )

        )


        self.register_buffer(

            "inv_freq",

            inv_freq,

            persistent=False,

        )


        self._build_cache(

            max_seq_len

        )


    def _build_cache(

        self,
        seq_len,

    ):

        positions = torch.arange(

            seq_len,

            dtype=torch.float32,

            device=self.inv_freq.device,

        )


        freqs = torch.outer(

            positions,

            self.inv_freq,

        )


        self.register_buffer(

            "cos_cached",

            freqs.cos(),

            persistent=False,

        )


        self.register_buffer(

            "sin_cached",

            freqs.sin(),

            persistent=False,

        )


    def forward(

        self,
        seq_len,
        device,

    ):

        if (

            seq_len

            >

            self.cos_cached.shape[0]

        ):

            self._build_cache(

                seq_len

            )


        return (

            self.cos_cached[

                :seq_len

            ].to(device),

            self.sin_cached[

                :seq_len

            ].to(device),

        )


def rotate_half(x):

    half = x.shape[-1] // 2

    x1 = x[..., :half]

    x2 = x[..., half:]

    return torch.cat(

        [-x2, x1],

        dim=-1,

    )


def apply_rope(

    q,
    k,
    cos,
    sin,

):

    cos = torch.cat(

        [
            cos,
            cos,
        ],

        dim=-1,

    )


    sin = torch.cat(

        [
            sin,
            sin,
        ],

        dim=-1,

    )


    cos = cos[
        None,
        None,
        :,
        :
    ]


    sin = sin[
        None,
        None,
        :,
        :
    ]


    q = (

        q * cos

        +

        rotate_half(q) * sin

    )


    k = (

        k * cos

        +

        rotate_half(k) * sin

    )


    return q, k


# ================================================================
# 13. FUSED SWIGLU
# ================================================================

class SwiGLU(nn.Module):

    def __init__(

        self,
        d_model,
        d_ff,

    ):

        super().__init__()


        # One GEMM instead of two.

        self.in_proj = nn.Linear(

            d_model,

            2 * d_ff,

            bias=False,

        )


        self.out_proj = nn.Linear(

            d_ff,

            d_model,

            bias=False,

        )


    def forward(self, x):

        gate, up = self.in_proj(x).chunk(

            2,

            dim=-1,

        )


        return self.out_proj(

            F.silu(gate) * up

        )


# ================================================================
# 14. GQA ATTENTION
# ================================================================

class GQAAttention(nn.Module):

    def __init__(

        self,

        d_model,

        n_q_heads,

        n_kv_heads,

        max_seq_len,

        window,

        is_global,

        rope_base,

        dropout=0.0,

    ):

        super().__init__()


        assert (

            d_model % n_q_heads == 0

        )


        assert (

            n_q_heads % n_kv_heads == 0

        )


        self.d_model = d_model

        self.n_q_heads = n_q_heads

        self.n_kv_heads = n_kv_heads

        self.head_dim = (

            d_model // n_q_heads

        )


        self.window = window

        self.is_global = is_global


        # --------------------------------------------------------
        # FUSED QKV PROJECTION
        #
        # Q = 16 × 64 = 1024
        # K =  4 × 64 = 256
        # V =  4 × 64 = 256
        #
        # Total = 1536
        # --------------------------------------------------------

        q_dim = (

            n_q_heads *

            self.head_dim

        )


        kv_dim = (

            n_kv_heads *

            self.head_dim

        )


        self.qkv_proj = nn.Linear(

            d_model,

            q_dim + kv_dim + kv_dim,

            bias=False,

        )


        self.out_proj = nn.Linear(

            d_model,

            d_model,

            bias=False,

        )


        # --------------------------------------------------------
        # RoPE
        # --------------------------------------------------------

        self.rope = RotaryEmbedding(

            self.head_dim,

            max_seq_len,

            rope_base,

        )


        self.dropout = dropout


        self.kv_repeat = (

            n_q_heads

            //

            n_kv_heads

        )


        # --------------------------------------------------------
        # Local causal mask
        # --------------------------------------------------------

        i = torch.arange(

            max_seq_len

        )[:, None]


        j = torch.arange(

            max_seq_len

        )[None, :]


        local_mask = (

            (j <= i)

            &

            (

                j >=

                i - window + 1

            )

        )


        self.register_buffer(

            "local_mask",

            local_mask,

            persistent=False,

        )


    def forward(self, x):

        B, T, C = x.shape


        # --------------------------------------------------------
        # FUSED QKV
        # --------------------------------------------------------

        qkv = self.qkv_proj(x)


        q_dim = (

            self.n_q_heads *

            self.head_dim

        )


        kv_dim = (

            self.n_kv_heads *

            self.head_dim

        )


        q, k, v = torch.split(

            qkv,

            [

                q_dim,

                kv_dim,

                kv_dim,

            ],

            dim=-1,

        )


        # --------------------------------------------------------
        # Q reshape
        # --------------------------------------------------------

        q = q.view(

            B,

            T,

            self.n_q_heads,

            self.head_dim,

        ).transpose(

            1,

            2,

        )


        # --------------------------------------------------------
        # K reshape
        # --------------------------------------------------------

        k = k.view(

            B,

            T,

            self.n_kv_heads,

            self.head_dim,

        ).transpose(

            1,

            2,

        )


        # --------------------------------------------------------
        # V reshape
        # --------------------------------------------------------

        v = v.view(

            B,

            T,

            self.n_kv_heads,

            self.head_dim,

        ).transpose(

            1,

            2,

        )


        # --------------------------------------------------------
        # RoPE
        # --------------------------------------------------------

        cos, sin = self.rope(

            T,

            x.device,

        )


        q, k = apply_rope(

            q,

            k,

            cos,

            sin,

        )


        # --------------------------------------------------------
        # Native GQA
        #
        # Q = 16 heads
        # K = 4 heads
        # V = 4 heads
        # --------------------------------------------------------

        dropout_p = (

            self.dropout

            if self.training

            else 0.0

        )


        if self.is_global:

            # ----------------------------------------------------
            # FULL CAUSAL ATTENTION
            # ----------------------------------------------------

            y = F.scaled_dot_product_attention(

                q,

                k,

                v,

                attn_mask=None,

                dropout_p=dropout_p,

                is_causal=True,

                enable_gqa=True,

            )

        else:

            # ----------------------------------------------------
            # LOCAL CAUSAL ATTENTION
            # ----------------------------------------------------

            mask = self.local_mask[

                :T,

                :T

            ]


            y = F.scaled_dot_product_attention(

                q,

                k,

                v,

                attn_mask=mask,

                dropout_p=dropout_p,

                is_causal=False,

                enable_gqa=True,

            )


        # --------------------------------------------------------
        # Merge heads
        # --------------------------------------------------------

        y = y.transpose(

            1,

            2,

        ).contiguous()


        y = y.view(

            B,

            T,

            C,

        )


        return self.out_proj(y)


# ================================================================
# 15. TRANSFORMER BLOCK
# ================================================================

class Arch4Block(nn.Module):

    def __init__(

        self,

        cfg,

        layer_idx,

    ):

        super().__init__()


        self.layer_idx = layer_idx


        self.is_global = (

            (layer_idx + 1)

            %

            cfg["GLOBAL_EVERY"]

            == 0

        )


        self.norm1 = nn.LayerNorm(

            cfg["D_MODEL"]

        )


        self.attn = GQAAttention(

            d_model=cfg["D_MODEL"],

            n_q_heads=cfg["N_Q_HEADS"],

            n_kv_heads=cfg["N_KV_HEADS"],

            max_seq_len=cfg["MAX_SEQ_LEN"],

            window=cfg["WINDOW"],

            is_global=self.is_global,

            rope_base=cfg["ROPE_BASE"],

            dropout=cfg["DROPOUT"],

        )


        self.norm2 = nn.LayerNorm(

            cfg["D_MODEL"]

        )


        self.ffn = SwiGLU(

            cfg["D_MODEL"],

            cfg["D_FF"],

        )


    def forward(self, x):

        # --------------------------------------------------------
        # Pre-LN attention
        # --------------------------------------------------------

        x = x + self.attn(

            self.norm1(x)

        )


        # --------------------------------------------------------
        # Pre-LN FFN
        # --------------------------------------------------------

        x = x + self.ffn(

            self.norm2(x)

        )


        return x


# ================================================================
# 16. ARCHITECTURE 4
# ================================================================

class Architecture4(nn.Module):

    def __init__(

        self,

        cfg,

    ):

        super().__init__()


        self.cfg = cfg


        # --------------------------------------------------------
        # Token embedding
        # --------------------------------------------------------

        self.token_embedding = nn.Embedding(

            cfg["VOCAB_SIZE"],

            cfg["D_MODEL"],

        )


        # --------------------------------------------------------
        # Transformer
        # --------------------------------------------------------

        self.layers = nn.ModuleList([

            Arch4Block(

                cfg,

                i,

            )

            for i in range(

                cfg["N_LAYERS"]

            )

        ])


        # --------------------------------------------------------
        # Final normalization
        # --------------------------------------------------------

        self.final_norm = nn.LayerNorm(

            cfg["D_MODEL"]

        )


        # --------------------------------------------------------
        # LM head
        # --------------------------------------------------------

        self.lm_head = nn.Linear(

            cfg["D_MODEL"],

            cfg["VOCAB_SIZE"],

            bias=False,

        )


        # --------------------------------------------------------
        # Tied weights
        # --------------------------------------------------------

        self.lm_head.weight = (

            self.token_embedding.weight

        )


        # --------------------------------------------------------
        # Initialization
        # --------------------------------------------------------

        self._init_weights()


    def _init_weights(self):

        std = self.cfg["INIT_STD"]


        # --------------------------------------------------------
        # Token embedding
        # --------------------------------------------------------

        nn.init.normal_(

            self.token_embedding.weight,

            mean=0.0,

            std=std,

        )


        # --------------------------------------------------------
        # Linear layers
        # --------------------------------------------------------

        for module in self.modules():

            if not isinstance(

                module,

                nn.Linear,

            ):

                continue


            # LM head is tied to embedding.

            if (

                module.weight

                is

                self.token_embedding.weight

            ):

                continue


            nn.init.normal_(

                module.weight,

                mean=0.0,

                std=std,

            )


            if module.bias is not None:

                nn.init.zeros_(

                    module.bias

                )


        # --------------------------------------------------------
        # Residual scaling
        # --------------------------------------------------------

        residual_scale = (

            1.0

            /

            math.sqrt(

                2 *

                self.cfg["N_LAYERS"]

            )

        )


        for block in self.layers:

            block.attn.out_proj.weight.data.mul_(

                residual_scale

            )


            block.ffn.out_proj.weight.data.mul_(

                residual_scale

            )


    def forward(

        self,

        input_ids,

        targets=None,

        return_logits=True,

    ):

        # --------------------------------------------------------
        # Embedding
        #
        # NO sqrt(d_model) scaling.
        # --------------------------------------------------------

        x = self.token_embedding(

            input_ids

        )


        # --------------------------------------------------------
        # Transformer
        # --------------------------------------------------------

        for block in self.layers:

            x = block(x)


        # --------------------------------------------------------
        # Final LN
        # --------------------------------------------------------

        x = self.final_norm(x)


        # --------------------------------------------------------
        # LM head
        # --------------------------------------------------------

        logits = self.lm_head(x)


        # --------------------------------------------------------
        # CE
        # --------------------------------------------------------

        loss = None


        if targets is not None:

            loss = F.cross_entropy(

                logits.reshape(

                    -1,

                    logits.size(-1),

                ),

                targets.reshape(

                    -1,

                ),

            )


        if not return_logits:

            return None, loss


        return logits, loss


# ================================================================
# 17. BUILD MODEL
# ================================================================

print(

    "\nBuilding Architecture 4..."

)


raw_model = Architecture4(

    CFG

).to(DEVICE)


# ================================================================
# 18. PARAMETER COUNT
# ================================================================

total_params = sum(

    p.numel()

    for p in raw_model.parameters()

)


trainable_params = sum(

    p.numel()

    for p in raw_model.parameters()

    if p.requires_grad

)


print(

    f"\nParameters: "
    f"{total_params:,}"

)


print(

    f"Trainable: "
    f"{trainable_params:,}"

)


print(

    f"Parameters (M): "
    f"{total_params / 1e6:.3f}M"

)


# ================================================================
# 19. ATTENTION PATTERN
# ================================================================

print(

    "\nAttention pattern:"

)


for i, block in enumerate(

    raw_model.layers,

    start=1,

):

    if block.is_global:

        print(

            f"Layer {i:2d}: GLOBAL"

        )

    else:

        print(

            f"Layer {i:2d}: "
            f"LOCAL W={CFG['WINDOW']}"

        )


# ================================================================
# 20. OPTIMIZER
# ================================================================

decay_params = []

no_decay_params = []


for name, param in raw_model.named_parameters():

    if not param.requires_grad:

        continue


    if (

        param.ndim >= 2

        and

        "norm" not in name.lower()

    ):

        decay_params.append(param)

    else:

        no_decay_params.append(param)


optimizer_kwargs = {

    "lr":
        CFG["LR"],

    "betas":
        (0.9, 0.95),

    "eps":
        1e-8,

}


use_fused = False


if (

    DEVICE.type == "cuda"

    and

    CFG["USE_FUSED_ADAMW"]

):

    try:

        optimizer = torch.optim.AdamW(

            [

                {

                    "params":
                        decay_params,

                    "weight_decay":
                        CFG["WEIGHT_DECAY"],

                },

                {

                    "params":
                        no_decay_params,

                    "weight_decay":
                        0.0,

                },

            ],

            fused=True,

            **optimizer_kwargs,

        )


        use_fused = True


        print(

            "\nOptimizer: FUSED AdamW"

        )


    except Exception as e:

        print(

            "\nFused AdamW unavailable."

        )

        print(

            "Falling back to foreach AdamW."

        )

        print(

            str(e)[:300]

        )


        optimizer = torch.optim.AdamW(

            [

                {

                    "params":
                        decay_params,

                    "weight_decay":
                        CFG["WEIGHT_DECAY"],

                },

                {

                    "params":
                        no_decay_params,

                    "weight_decay":
                        0.0,

                },

            ],

            foreach=True,

            **optimizer_kwargs,

        )

else:

    optimizer = torch.optim.AdamW(

        [

            {

                "params":
                    decay_params,

                "weight_decay":
                    CFG["WEIGHT_DECAY"],

            },

            {

                "params":
                    no_decay_params,

                "weight_decay":
                    0.0,

            },

        ],

        foreach=True,

        **optimizer_kwargs,

    )


# ================================================================
# 21. AMP
# ================================================================

USE_AMP = (

    DEVICE.type == "cuda"

)


scaler = torch.amp.GradScaler(

    "cuda",

    enabled=USE_AMP,

)


# ================================================================
# 22. LEARNING RATE
# ================================================================

def get_lr(step):

    if step < CFG["WARMUP_STEPS"]:

        return (

            CFG["LR"]

            *

            (step + 1)

            /

            CFG["WARMUP_STEPS"]

        )


    progress = (

        step

        -

        CFG["WARMUP_STEPS"]

    ) / (

        CFG["MAX_STEPS"]

        -

        CFG["WARMUP_STEPS"]

    )


    progress = min(

        max(progress, 0.0),

        1.0,

    )


    cosine = (

        0.5

        *

        (

            1.0

            +

            math.cos(

                math.pi * progress

            )

        )

    )


    min_lr = (

        CFG["LR"]

        *

        CFG["MIN_LR_RATIO"]

    )


    return (

        min_lr

        +

        (

            CFG["LR"]

            -

            min_lr

        )

        *

        cosine

    )


def set_lr(

    optimizer,

    lr,

):

    for group in optimizer.param_groups:

        group["lr"] = lr


# ================================================================
# 23. CUDA GRAPH ITERATION MARKER
# ================================================================

def mark_cudagraph_iteration():

    """

    Mark the beginning of one logical iteration when using
    torch.compile(reduce-overhead).

    We intentionally call this ONCE per optimizer step,
    before gradient accumulation begins.

    """

    if not compile_enabled:

        return


    if not hasattr(

        torch,

        "compiler"

    ):

        return


    if not hasattr(

        torch.compiler,

        "cudagraph_mark_step_begin"

    ):

        return


    torch.compiler.cudagraph_mark_step_begin()


# ================================================================
# 24. MEMORY REPORT
# ================================================================

def memory_report():

    if DEVICE.type != "cuda":

        return


    allocated = (

        torch.cuda.memory_allocated()

        /

        1024**3

    )


    reserved = (

        torch.cuda.memory_reserved()

        /

        1024**3

    )


    peak = (

        torch.cuda.max_memory_allocated()

        /

        1024**3

    )


    print(

        f"GPU memory | "

        f"allocated={allocated:.2f} GB | "

        f"reserved={reserved:.2f} GB | "

        f"peak={peak:.2f} GB"

    )


# ================================================================
# 25. INITIALIZATION TEST
# ================================================================

compile_enabled = False


print(

    "\n" + "=" * 70

)

print(

    "INITIALIZATION TEST"

)

print(

    "=" * 70

)


raw_model.eval()


gc.collect()

torch.cuda.empty_cache()


x, y = sample_batch(

    train_arrays,

    1,

    CFG["SEQ_LEN"],

)


x = x.to(

    DEVICE,

    non_blocking=True,

)


y = y.to(

    DEVICE,

    non_blocking=True,

)


with torch.no_grad():

    with torch.amp.autocast(

        "cuda",

        enabled=USE_AMP,

        dtype=torch.float16,

    ):

        logits, loss = raw_model(

            x,

            y,

        )


logits_f = logits.float()


print(

    "Initial loss:",

    f"{loss.item():.4f}"

)


print(

    "Expected random CE:",

    f"{math.log(CFG['VOCAB_SIZE']):.4f}"

)


print(

    "Logits mean:",

    f"{logits_f.mean().item():.4f}"

)


print(

    "Logits std:",

    f"{logits_f.std().item():.4f}"

)


print(

    "Logits min:",

    f"{logits_f.min().item():.4f}"

)


print(

    "Logits max:",

    f"{logits_f.max().item():.4f}"

)


memory_report()


del logits

del logits_f

del loss

del x

del y


gc.collect()

torch.cuda.empty_cache()


# ================================================================
# 26. TEST BATCH SIZE
# ================================================================

print(

    "\n" + "=" * 70

)

print(

    "TESTING TARGET BATCH SIZE"

)

print(

    "=" * 70

)


def test_batch_size(batch_size):

    raw_model.train()


    optimizer.zero_grad(

        set_to_none=True

    )


    gc.collect()

    torch.cuda.empty_cache()


    if DEVICE.type == "cuda":

        torch.cuda.reset_peak_memory_stats()


    x, y = sample_batch(

        train_arrays,

        batch_size,

        CFG["SEQ_LEN"],

    )


    x = x.to(

        DEVICE,

        non_blocking=True,

    )


    y = y.to(

        DEVICE,

        non_blocking=True,

    )


    try:

        # For this isolated test, this is one logical
        # training iteration.

        if hasattr(

            torch,

            "compiler"

        ) and hasattr(

            torch.compiler,

            "cudagraph_mark_step_begin"

        ):

            # Raw/eager model currently, so there is no
            # need to mark it. This is intentionally omitted.

            pass


        with torch.amp.autocast(

            "cuda",

            enabled=USE_AMP,

            dtype=torch.float16,

        ):

            logits, loss = raw_model(

                x,

                y,

            )


        scaler.scale(

            loss

        ).backward()


        if DEVICE.type == "cuda":

            torch.cuda.synchronize()


        peak = (

            torch.cuda.max_memory_allocated()

            /

            1024**3

        )


        optimizer.zero_grad(

            set_to_none=True

        )


        del logits

        del loss

        del x
        del y


        gc.collect()

        torch.cuda.empty_cache()


        return True, peak


    except torch.cuda.OutOfMemoryError:

        print(

            f"BATCH {batch_size} "
            f"caused CUDA OOM."

        )


        optimizer.zero_grad(

            set_to_none=True

        )


        for var_name in (

            "logits",
            "loss",
            "x",
            "y",

        ):

            if var_name in locals():

                try:

                    del locals()[var_name]

                except Exception:

                    pass


        gc.collect()

        torch.cuda.empty_cache()


        return False, None


# ---------------------------------------------------------------
# Try B8.
# ---------------------------------------------------------------

b8_ok, b8_peak = test_batch_size(

    8

)


if b8_ok:

    CFG["BATCH_SIZE"] = 8

    CFG["GRAD_ACCUM"] = 2

    CFG["BATCH_SIZE"] = 8

    CFG["GRAD_ACCUM"] = 2


    print(

        "\nBATCH 8 PASSED."

    )


    print(

        f"Peak memory: "
        f"{b8_peak:.2f} GB"

    )


else:

    # -----------------------------------------------------------
    # B4 fallback
    # -----------------------------------------------------------

    b4_ok, b4_peak = test_batch_size(

        4

    )


    if not b4_ok:

        raise RuntimeError(

            "Batch 8 and Batch 4 both "
            "caused CUDA OOM."

        )


    CFG["BATCH_SIZE"] = 4

    CFG["GRAD_ACCUM"] = 4


    print(

        "\nUsing fallback: "
        "BATCH 4 / ACCUM 4"

    )


    print(

        f"Peak memory: "
        f"{b4_peak:.2f} GB"

    )


# ---------------------------------------------------------------
# Verify effective tokens.
# ---------------------------------------------------------------

tokens_per_microbatch = (

    CFG["BATCH_SIZE"]

    *

    CFG["SEQ_LEN"]

)


tokens_per_step = (

    tokens_per_microbatch

    *

    CFG["GRAD_ACCUM"]

)


assert (

    tokens_per_step

    ==

    16384

)


print(

    "\nSelected configuration:"

)


print(

    f"Batch size: "
    f"{CFG['BATCH_SIZE']}"

)


print(

    f"Grad accumulation: "
    f"{CFG['GRAD_ACCUM']}"

)


print(

    f"Tokens / microbatch: "
    f"{tokens_per_microbatch:,}"

)


print(

    f"Tokens / optimizer step: "
    f"{tokens_per_step:,}"

)


# ================================================================
# 27. TORCH.COMPILE
# ================================================================

model = raw_model


if (

    CFG["USE_COMPILE"]

    and

    hasattr(torch, "compile")

    and

    DEVICE.type == "cuda"

):

    print(

        "\n" + "=" * 70

    )

    print(

        "COMPILING MODEL"

    )

    print(

        "=" * 70

    )


    try:

        model = torch.compile(

            raw_model,

            mode=CFG["COMPILE_MODE"],

            dynamic=False,

        )


        # --------------------------------------------------------
        # COMPILE WARMUP
        #
        # One forward + backward is one logical iteration.
        # --------------------------------------------------------

        optimizer.zero_grad(

            set_to_none=True

        )


        raw_model.train()


        x, y = sample_batch(

            train_arrays,

            CFG["BATCH_SIZE"],

            CFG["SEQ_LEN"],

        )


        x = x.to(

            DEVICE,

            non_blocking=True,

        )


        y = y.to(

            DEVICE,

            non_blocking=True,


        )


        # IMPORTANT:
        # Start a new CUDAGraph iteration here.

        mark_cudagraph_iteration()


        with torch.amp.autocast(

            "cuda",

            enabled=USE_AMP,

            dtype=torch.float16,

        ):

            logits, loss = model(

                x,

                y,

            )


        scaler.scale(

            loss

        ).backward()


        if DEVICE.type == "cuda":

            torch.cuda.synchronize()


        optimizer.zero_grad(

            set_to_none=True

        )


        del logits

        del loss

        del x

        del y


        gc.collect()

        torch.cuda.empty_cache()


        compile_enabled = True


        print(

            "torch.compile: ENABLED"

        )


    except Exception as e:

        print(

            "\ntorch.compile warmup failed."

        )


        print(

            "Falling back to eager PyTorch."

        )


        print(

            "Reason:",

            str(e)[:1000]

        )


        optimizer.zero_grad(

            set_to_none=True

        )


        model = raw_model

        compile_enabled = False


        gc.collect()

        torch.cuda.empty_cache()


else:

    print(

        "\ntorch.compile: DISABLED"

    )


# ================================================================
# 28. TRAINING SUMMARY
# ================================================================

tokens_per_microbatch = (

    CFG["BATCH_SIZE"]

    *

    CFG["SEQ_LEN"]

)


tokens_per_step = (

    tokens_per_microbatch

    *

    CFG["GRAD_ACCUM"]

)


total_training_tokens = (

    tokens_per_step

    *

    CFG["MAX_STEPS"]

)


print(

    "\n" + "=" * 70

)

print(

    "ARCHITECTURE 4 — FINAL TRAINING CONFIG"

)

print(

    "=" * 70

)


print(

    f"Parameters: "
    f"{total_params:,}"

)


print(

    f"Parameters (M): "
    f"{total_params / 1e6:.3f}"

)


print(

    f"Batch: "
    f"{CFG['BATCH_SIZE']}"

)


print(

    f"Gradient accumulation: "
    f"{CFG['GRAD_ACCUM']}"

)


print(

    f"Sequence length: "
    f"{CFG['SEQ_LEN']}"

)


print(

    f"Tokens / microbatch: "
    f"{tokens_per_microbatch:,}"

)


print(

    f"Tokens / optimizer step: "
    f"{tokens_per_step:,}"

)


print(

    f"Total training tokens: "
    f"{total_training_tokens:,}"

)


print(

    f"Fused AdamW: "
    f"{use_fused}"

)


print(

    f"torch.compile: "
    f"{compile_enabled}"

)


# ================================================================
# 29. CHECKPOINT FUNCTION
# ================================================================

def save_checkpoint(

    step,

    best_val,

):

    path = os.path.join(

        CFG["OUTPUT_DIR"],

        f"checkpoint_{step:05d}.pt",

    )


    torch.save(

        {

            "step":
                step,

            "model":
                raw_model.state_dict(),

            "optimizer":
                optimizer.state_dict(),

            "scaler":
                scaler.state_dict(),

            "best_val":
                best_val,

            "cfg":
                CFG,

        },

        path,

    )


    print(

        f"\nSaved checkpoint:\n{path}"

    )


# ================================================================
# 30. VALIDATION
# ================================================================

@torch.no_grad()

def evaluate_validation(

    n_batches=30,

):

    model.eval()


    source_losses = {

        "acts": [],

        "sc": [],

        "tribunals": [],

    }


    batches_per_source = max(

        1,

        n_batches //

        len(SOURCE_NAMES)

    )


    for source_name in SOURCE_NAMES:

        data = val_arrays[

            source_name

        ]


        for _ in range(

            batches_per_source

        ):


            batch_size = CFG["BATCH_SIZE"]


            xs = np.empty(

                (

                    batch_size,

                    CFG["SEQ_LEN"],

                ),

                dtype=np.int64,

            )


            ys = np.empty(

                (

                    batch_size,

                    CFG["SEQ_LEN"],

                ),

                dtype=np.int64,

            )


            max_start = (

                len(data)

                -

                CFG["SEQ_LEN"]

                -

                1

            )


            starts = np.random.randint(

                0,

                max_start + 1,

                size=batch_size,

            )


            for b, start in enumerate(

                starts

            ):

                xs[b] = data[

                    start:

                    start + CFG["SEQ_LEN"]

                ]


                ys[b] = data[

                    start + 1:

                    start + CFG["SEQ_LEN"] + 1

                ]


            x = torch.from_numpy(

                xs

            ).to(

                DEVICE,

                non_blocking=True,

            )


            y = torch.from_numpy(

                ys

            ).to(

                DEVICE,

                non_blocking=True,

            )


            # Each validation call is one inference iteration.

            if compile_enabled:

                mark_cudagraph_iteration()


            with torch.amp.autocast(

                "cuda",

                enabled=USE_AMP,

                dtype=torch.float16,

            ):

                _, loss = model(

                    x,

                    y,

                    return_logits=False,

                )


            source_losses[

                source_name

            ].append(

                loss.item()

            )


            del x

            del y

            del loss


    model.train()


    result = {}


    for source_name in SOURCE_NAMES:

        result[source_name] = float(

            np.mean(

                source_losses[

                    source_name

                ]

            )

        )


    # ------------------------------------------------------------
    # Same mixture-weighted validation criterion.
    # ------------------------------------------------------------

    result["mean"] = (

        SOURCE_PROBS[0]

        *

        result["acts"]

        +

        SOURCE_PROBS[1]

        *

        result["sc"]

        +

        SOURCE_PROBS[2]

        *

        result["tribunals"]

    )


    return result


# ================================================================
# 31. SPEED WARMUP
# ================================================================

print(

    "\n" + "=" * 70

)

print(

    "TRAINING SPEED WARMUP"

)

print(

    "=" * 70

)


model.train()


warmup_steps = 2


optimizer.zero_grad(

    set_to_none=True

)


warmup_start = time.time()


for warm_step in range(

    warmup_steps

):

    optimizer.zero_grad(

        set_to_none=True

    )


    # ------------------------------------------------------------
    # IMPORTANT:
    # One CUDA Graph logical iteration begins here.
    # There are multiple compiled calls inside it due to
    # gradient accumulation.
    # ------------------------------------------------------------

    mark_cudagraph_iteration()


    for micro_step in range(

        CFG["GRAD_ACCUM"]

    ):

        x, y = sample_batch(

            train_arrays,

            CFG["BATCH_SIZE"],

            CFG["SEQ_LEN"],

        )


        x = x.to(

            DEVICE,

            non_blocking=True,

        )


        y = y.to(

            DEVICE,

            non_blocking=True,

        )


        with torch.amp.autocast(

            "cuda",

            enabled=USE_AMP,

            dtype=torch.float16,

        ):

            logits, loss = model(

                x,

                y,

            )


            scaled_loss = (

                loss

                /

                CFG["GRAD_ACCUM"]

            )


        scaler.scale(

            scaled_loss

        ).backward()


        del logits

        del loss

        del scaled_loss

        del x

        del y


    optimizer.zero_grad(

        set_to_none=True

    )


if DEVICE.type == "cuda":

    torch.cuda.synchronize()


warmup_elapsed = (

    time.time()

    -

    warmup_start

)


warmup_tokens = (

    warmup_steps

    *

    tokens_per_step

)


warmup_tok_sec = (

    warmup_tokens

    /

    max(

        warmup_elapsed,

        1e-6

    )

)


print(

    f"Warmup throughput: "
    f"{warmup_tok_sec:,.0f} tok/s"

)


memory_report()


# ================================================================
# 32. FINAL CLEANUP BEFORE TRAINING
# ================================================================

optimizer.zero_grad(

    set_to_none=True

)


gc.collect()

torch.cuda.empty_cache()


if DEVICE.type == "cuda":

    torch.cuda.reset_peak_memory_stats()


model.train()


# ================================================================
# 33. CHECKPOINT SETTINGS
# ================================================================

best_val = float("inf")


start_time = time.time()


# ================================================================
# 34. TRAINING LOOP
# ================================================================

print(

    "\n" + "=" * 70

)

print(

    "STARTING ARCHITECTURE 4 TRAINING"

)

print(

    "=" * 70

)


for step in range(

    1,

    CFG["MAX_STEPS"] + 1,

):


    # ------------------------------------------------------------
    # One logical optimizer iteration begins here.
    #
    # THIS IS THE IMPORTANT CUDAGRAPH FIX.
    # ------------------------------------------------------------

    mark_cudagraph_iteration()


    optimizer.zero_grad(

        set_to_none=True

    )


    total_loss = 0.0


    # ------------------------------------------------------------
    # Gradient accumulation
    # ------------------------------------------------------------

    for micro_step in range(

        CFG["GRAD_ACCUM"]

    ):


        x, y = sample_batch(

            train_arrays,

            CFG["BATCH_SIZE"],

            CFG["SEQ_LEN"],

        )


        x = x.to(

            DEVICE,

            non_blocking=True,

        )


        y = y.to(

            DEVICE,

            non_blocking=True,

        )


        with torch.amp.autocast(

            "cuda",

            enabled=USE_AMP,

            dtype=torch.float16,

        ):

            logits, loss = model(

                x,

                y,

            )


            scaled_loss = (

                loss

                /

                CFG["GRAD_ACCUM"]

            )


        scaler.scale(

            scaled_loss

        ).backward()


        total_loss += loss.item()


        del logits

        del loss

        del scaled_loss

        del x

        del y


    # ------------------------------------------------------------
    # Unscale
    # ------------------------------------------------------------

    scaler.unscale_(

        optimizer

    )


    # ------------------------------------------------------------
    # Gradient clipping
    # ------------------------------------------------------------

    grad_norm = (

        torch.nn.utils.clip_grad_norm_(

            raw_model.parameters(),

            CFG["GRAD_CLIP"],

        )

    )


    # ------------------------------------------------------------
    # Learning rate
    # ------------------------------------------------------------

    lr = get_lr(

        step - 1

    )


    set_lr(

        optimizer,

        lr

    )


    # ------------------------------------------------------------
    # Optimizer update
    # ------------------------------------------------------------

    scaler.step(

        optimizer

    )


    scaler.update()


    # ------------------------------------------------------------
    # Logging
    # ------------------------------------------------------------

    if (

        step == 1

        or

        step % 50 == 0

    ):


        elapsed = (

            time.time()

            -

            start_time

        )


        effective_tok_sec = (

            step

            *

            tokens_per_step

            /

            max(

                elapsed,

                1e-6

            )

        )


        avg_loss = (

            total_loss

            /

            CFG["GRAD_ACCUM"]

        )


        print(

            f"STEP "

            f"{step:5d}/"

            f"{CFG['MAX_STEPS']} | "

            f"loss "

            f"{avg_loss:.4f} | "

            f"lr "

            f"{lr:.3e} | "

            f"grad "

            f"{float(grad_norm):.3f} | "

            f"{effective_tok_sec:,.0f} tok/s"

        )


    # ------------------------------------------------------------
    # Validation
    # ------------------------------------------------------------

    if (

        step

        %

        CFG["EVAL_EVERY"]

        == 0

    ):


        val = evaluate_validation(

            CFG["EVAL_BATCHES"]

        )


        print(

            "\n"

            f"STEP {step} VALIDATION\n"

            f"  weighted mean = "
            f"{val['mean']:.4f}\n"

            f"  acts          = "
            f"{val['acts']:.4f}\n"

            f"  sc            = "
            f"{val['sc']:.4f}\n"

            f"  tribunals     = "
            f"{val['tribunals']:.4f}\n"

        )


        memory_report()


        # --------------------------------------------------------
        # Best checkpoint
        # --------------------------------------------------------

        if (

            val["mean"]

            <

            best_val

        ):


            best_val = val["mean"]


            best_path = os.path.join(

                CFG["OUTPUT_DIR"],

                "best.pt",

            )


            torch.save(

                {

                    "step":
                        step,

                    "model":
                        raw_model.state_dict(),

                    "optimizer":
                        optimizer.state_dict(),

                    "scaler":
                        scaler.state_dict(),

                    "best_val":
                        best_val,

                    "cfg":
                        CFG,

                },

                best_path,

            )


            print(

                "NEW BEST CHECKPOINT:",

                best_path

            )


    # ------------------------------------------------------------
    # Periodic checkpoint
    # ------------------------------------------------------------

    if (

        step

        %

        CFG["SAVE_EVERY"]

        == 0

    ):


        save_checkpoint(

            step,

            best_val

        )


# ================================================================
# 35. FINAL CHECKPOINT
# ================================================================

final_path = os.path.join(

    CFG["OUTPUT_DIR"],

    "latest.pt",

)


torch.save(

    {

        "step":
            CFG["MAX_STEPS"],

        "model":
            raw_model.state_dict(),

        "optimizer":
            optimizer.state_dict(),

        "scaler":
            scaler.state_dict(),

        "best_val":
            best_val,

        "cfg":
            CFG,

    },

    final_path,

)


# ================================================================
# 36. FINAL REPORT
# ================================================================

elapsed = (

    time.time()

    -

    start_time

)


average_tok_sec = (

    total_training_tokens

    /

    max(

        elapsed,

        1e-6

    )

)


print(

    "\n" + "=" * 70

)

print(

    "TRAINING COMPLETE"

)

print(

    "=" * 70

)


print(

    f"Parameters: "
    f"{total_params:,}"

)


print(

    f"Parameters (M): "
    f"{total_params / 1e6:.3f}"

)


print(

    f"Best validation: "
    f"{best_val:.4f}"

)


print(

    f"Training time: "
    f"{elapsed / 3600:.2f} hours"

)


print(

    f"Average throughput: "
    f"{average_tok_sec:,.0f} tok/s"

)


print(

    f"Final checkpoint:\n"
    f"{final_path}"

)


print(

    f"Best checkpoint:\n"
    f"{os.path.join(CFG['OUTPUT_DIR'], 'best.pt')}"

)


memory_report()

Device: cuda
GPU: Tesla T4
GPU memory: 14.56 GB

Checking files...
acts_train           True /kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/acts_train.bin
sc_train             True /kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/sc_train.bin
tribunals_train      True /kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/tribunals_train_f.bin
acts_val             True /kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/acts_val.bin
sc_val               True /kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/sc_val.bin
tribunals_val        True /kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/tribunals_val_f.bin
tokenizer            True /kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/tokenizer.json
test                 True /kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/test.json

Loading tokenizer...
Tokenizer loaded.

Training data:
acts        : 145,554,561 tokens
sc          : 249,782,556 tokens
tribunals   

W1005 23:53:52.352000 24 torch/_inductor/utils.py:1731] [0/0] Not enough SMs to use max_autotune_gemm mode


torch.compile: ENABLED

ARCHITECTURE 4 — FINAL TRAINING CONFIG
Parameters: 195,151,872
Parameters (M): 195.152
Batch: 4
Gradient accumulation: 4
Sequence length: 1024
Tokens / microbatch: 4,096
Tokens / optimizer step: 16,384
Total training tokens: 163,840,000
Fused AdamW: True
torch.compile: True

TRAINING SPEED WARMUP
Warmup throughput: 6,914 tok/s
GPU memory | allocated=0.75 GB | reserved=8.36 GB | peak=8.43 GB

STARTING ARCHITECTURE 4 TRAINING
STEP     1/10000 | loss 10.7852 | lr 6.000e-07 | grad 17.432 | 8,235 tok/s
STEP    50/10000 | loss 7.9917 | lr 3.000e-05 | grad 4.514 | 8,515 tok/s
STEP   100/10000 | loss 6.4977 | lr 6.000e-05 | grad 2.160 | 8,153 tok/s
STEP   150/10000 | loss 5.8957 | lr 9.000e-05 | grad 2.503 | 8,074 tok/s
STEP   200/10000 | loss 5.3612 | lr 1.200e-04 | grad 1.637 | 8,033 tok/s
STEP   250/10000 | loss 5.3463 | lr 1.500e-04 | grad 1.928 | 8,009 tok/s
STEP   300/10000 | loss 4.7501 | lr 1.800e-04 | grad 2.044 | 8,008 tok/s
STEP   350/10000 | loss 4.8289 | lr

RuntimeError: [enforce fail at inline_container.cc:672] . unexpected pos 2184202880 vs 2184202768